In [1]:
import os
import sys
import glob
import numpy as np

if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from utils import compute_Ve_over_cell

from dlutils.morpho import Tree
from dlutils.graphics import plot_tree

from neuron import h
h.load_file('stdrun.hoc')
h.celsius = 34

ModuleNotFoundError: No module named 'numpy'

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm, TwoSlopeNorm
from matplotlib.patches import Polygon
import seaborn as sns

fontsize = 9
lw = 0.75
matplotlib.rc('font', **{'family': 'Arial', 'size': fontsize})
matplotlib.rc('axes', **{'linewidth': 0.75, 'labelsize': fontsize})
matplotlib.rc('xtick', **{'labelsize': fontsize})
matplotlib.rc('ytick', **{'labelsize': fontsize})
matplotlib.rc('xtick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.minor', **{'width': lw, 'size':1.5})

In [ ]:
import dbbs_models
cell_type = 'granule'
cell = getattr(dbbs_models, f'build_{cell_type}_cell')()

In [ ]:
data_dir = os.path.join('..','..','dbbs-lab','models','dbbs_models','morphologies')
swc_file = [f for f in sorted(glob.glob(os.path.join(data_dir, '*.swc'))) if cell_type in f.lower()][0]
tree = Tree(swc_file)
print(f'SWC file: {swc_file}')

In [ ]:
Emag = 600 # [V/m]
θ    = 0*(-np.pi)
ϕ    = 0*np.pi/2
spherical_to_cartesian = lambda r,theta,phi: np.array([r*np.sin(theta)*np.cos(phi),
                                                       r*np.sin(theta)*np.sin(phi),
                                                       r*np.cos(theta)])
Efield = lambda X: spherical_to_cartesian(Emag,θ,ϕ)

In [ ]:
Ve,cell_secs,points = compute_Ve_over_cell(cell, Efield, full_output=True)
assert all([ve.size==sec.nseg for ve,sec in zip(Ve,cell_secs)])
N_secs = len(cell_secs)
data = np.vstack(list(points.values()))
PTS,V = data[:,:3],data[:,-1]*1e3

In [ ]:
%matplotlib inline
norm = TwoSlopeNorm(vmin=V.min(), vcenter=0, vmax=V.max())
height = 5
width = height * max([0.5, tree.xz_ratio])
fig,ax = plt.subplots(1, 1, figsize=(width,height))
plot_tree(tree, coords='xz', cmap='coolwarm', norm=norm, points=PTS, values=V,
          show_cbar=True, cbar_ticks=5, cbar_label=r'$V_{\mathrm{extra}}$ (mV)', ax=ax, diam_coeff=10)
ax.axis('off')
fig.tight_layout()
plt.savefig(f'{cell_type}_Vextra_morpho.pdf')

In [ ]:
delay = 50   # [ms]
after = 50   # [ms]
stim_dur = 1 # [ms]

dt = min(h.dt, stim_dur/5)
tstop = delay + stim_dur + after
time = np.r_[0 : tstop : dt]
stim = np.zeros_like(time)
idx = (time >= delay) & (time < delay+stim_dur)
stim[idx] = 1
t_vec = h.Vector(time)
E_vecs = [[] for _ in range(N_secs)]
for i,sec in enumerate(cell_secs):
    sec.insert('extracellular')
    for j,seg in enumerate(sec):
        vec = h.Vector(stim * Ve[i][j] * 1e3)
        vec.play(seg._ref_e_extracellular, t_vec, 1)
        E_vecs[i].append(vec)
h.dt = dt

In [ ]:
rec = {'time': h.Vector(), 'Vsoma': h.Vector()}
rec.update({f'Vaxon-{i}-{j}': h.Vector() for i,sec in enumerate(cell.axon) for j in range(sec.nseg)})
rec['time'].record(h._ref_t)
rec['Vsoma'].record(cell.soma[0](0.5)._ref_v)
for i,sec in enumerate(cell.axon):
    for j,seg in enumerate(sec):
        rec[f'Vaxon-{i}-{j}'].record(seg._ref_v)

In [ ]:
h.tstop = tstop
h.cvode_active(0)
h.run()

In [ ]:
t,Vsoma = np.array(rec['time']), np.array(rec['Vsoma'])

In [ ]:
xy = np.array([[delay,-70],[delay+stim_dur,-70],[delay+stim_dur,40],[delay,40]])
fig,ax = plt.subplots(1, 1, figsize=(5,3.5))
poly = Polygon(xy, closed=True, fc=[1,.5,0], ec=[.8,.4,0], alpha=0.25)
idx = (t>delay-2) & (t<delay+10)
ax.plot(t[idx], Vsoma[idx], 'k', lw=2)
ax.add_patch(poly)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Vm (mV)')
ax.grid(which='major', axis='y', lw=0.5, ls=':', color=[.6,.6,.6])
ax.set_ylim([xy[:,1].min(), xy[:,1].max()])
sns.despine()
fig.tight_layout()
plt.savefig(f'{cell_type}_efield.pdf')